In [3]:
# !pip install fvcore

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from PIL import ImageFilter
import os
import cv2
import torch.nn.functional as F
import torchvision.transforms as tf
import shutil
import random
import gc
from transformers import SegformerForSemanticSegmentation, SegformerConfig

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Enable memory efficient mode
torch.cuda.empty_cache()
torch.backends.cudnn.benchmark = True

# Accuracy calculation (unchanged)
def cal_acc(pred_folder, gt_folder, classes=19):
    class AverageMeter(object):
        def __init__(self):
            self.reset()
        def reset(self):
            self.val, self.avg, self.sum, self.count = 0, 0, 0, 0
        def update(self, val, n=1):
            self.val = val
            self.sum += val * n
            self.count += n
            self.avg = self.sum / self.count
    def intersectionAndUnion(output, target, K, ignore_index=255):
        assert output.ndim in [1, 2, 3]
        assert output.shape == target.shape
        output = output.reshape(output.size).copy()
        target = target.reshape(target.size)
        output[np.where(target == ignore_index)[0]] = ignore_index
        intersection = output[np.where(output == target)[0]]
        area_intersection, _ = np.histogram(intersection, bins=np.arange(K + 1))
        area_output, _ = np.histogram(output, bins=np.arange(K + 1))
        area_target, _ = np.histogram(target, bins=np.arange(K + 1))
        area_union = area_output + area_target - area_intersection
        return area_intersection, area_union, area_target
    data_list = os.listdir(gt_folder)
    intersection_meter = AverageMeter()
    union_meter = AverageMeter()
    target_meter = AverageMeter()
    for image_name in data_list:
        pred = cv2.imread(os.path.join(pred_folder, image_name), cv2.IMREAD_GRAYSCALE)
        target = cv2.imread(os.path.join(gt_folder, image_name), cv2.IMREAD_GRAYSCALE)
        intersection, union, target = intersectionAndUnion(pred, target, classes)
        intersection_meter.update(intersection)
        union_meter.update(union)
        target_meter.update(target)
    iou_class = intersection_meter.sum / (union_meter.sum + 1e-10)
    mIoU = np.mean(iou_class)
    print(f'Eval result: mIoU {mIoU:.4f}.')
    return mIoU

def make_folder(dir_name):
    if not os.path.exists(dir_name):
        os.makedirs(dir_name)

def move_folders(grey_temp, color_temp, grey_rs, color_rs):
    if os.path.exists(grey_temp):
        make_folder(grey_rs)
        for file in os.listdir(grey_temp):
            shutil.move(os.path.join(grey_temp, file), os.path.join(grey_rs, file))
        if os.path.exists(grey_temp):
            shutil.rmtree(grey_temp)
    if os.path.exists(color_temp):
        make_folder(color_rs)
        for file in os.listdir(color_temp):
            shutil.move(os.path.join(color_temp, file), os.path.join(color_rs, file))
        if os.path.exists(color_temp):
            shutil.rmtree(color_temp)

def colorize(gray, palette):
    color = Image.fromarray(gray.astype(np.uint8)).convert('P')
    color.putpalette(palette)
    return color

def get_predictions(segNet, dataFolder, device):
    gray_folder, color_folder = './temp_grey', './temp_color'
    listImages, gt_folder = os.listdir(os.path.join(dataFolder, "testing/image")), os.path.join(dataFolder, "testing/label")
    print('Begin testing')
    make_folder(gray_folder)
    make_folder(color_folder)
    colors_path = os.path.join(dataFolder, "colors.txt")
    colors = np.loadtxt(colors_path).astype('uint8')
    
    transformTest = tf.Compose([tf.ToPILImage(), tf.Resize((375, 1242)), tf.ToTensor(),
                                tf.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
    
    for idx in range(len(listImages)):
        img_path = os.path.join(dataFolder, "testing/image", listImages[idx])
        img = cv2.imread(img_path)[:, :, 0:3]
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        original_height, original_width = img.shape[:2]

        if original_width <= 0 or original_height <= 0:
            print(f"Skipping image {listImages[idx]} due to invalid dimensions")
            continue

        img = transformTest(img).unsqueeze(0)
        
        with torch.no_grad():
            pred = segNet(img.to(device)).logits
            pred = F.interpolate(pred, size=(original_height, original_width), mode='bilinear', align_corners=False)
            gray = pred.argmax(1).squeeze(0).cpu().numpy().astype(np.uint8)
        
        color = colorize(gray, colors)
        gray_path = os.path.join(gray_folder, listImages[idx])
        color_path = os.path.join(color_folder, listImages[idx])
        cv2.imwrite(gray_path, gray)
        color.save(color_path)
        
        del pred
        torch.cuda.empty_cache()
        
    return gray_folder, color_folder

class ExpDataSet(Dataset):
    def __init__(self, dataFolder, is_train=True):
        self.is_train = is_train
        self.image_path = sorted(os.listdir(os.path.join(dataFolder, "training/image")))
        self.label_path = sorted(os.listdir(os.path.join(dataFolder, "training/label")))
        print(f'load info for {len(self.image_path)} images')
        assert len(self.image_path) == 150
        
        for idx in range(len(self.image_path)):
            assert self.image_path[idx] == self.label_path[idx]
            self.image_path[idx] = os.path.join(dataFolder, "training/image", self.image_path[idx])
            self.label_path[idx] = os.path.join(dataFolder, "training/label", self.label_path[idx])
        
        self.height = 375
        self.width = 1242  # Training input size updated per request
        
    def __getitem__(self, idx):
        img = cv2.imread(self.image_path[idx])[:, :, 0:3]
        label = cv2.imread(self.label_path[idx], 0)
        
        img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        label = Image.fromarray(label)
        
        if self.is_train:
            scale = random.choice([0.75, 1.0, 1.25])
            new_h = int(self.height * scale)
            new_w = int(self.width * scale)
            
            img = img.resize((new_w, new_h), Image.BILINEAR)
            label = label.resize((new_w, new_h), Image.NEAREST)
            
            if scale > 1.0:
                i = random.randint(0, new_h - self.height)
                j = random.randint(0, new_w - self.width)
                img = img.crop((j, i, j + self.width, i + self.height))
                label = label.crop((j, i, j + self.width, i + self.height))
            else:
                img = img.resize((self.width, self.height), Image.BILINEAR)
                label = label.resize((self.width, self.height), Image.NEAREST)
            
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                label = label.transpose(Image.FLIP_LEFT_RIGHT)
            
            color_jitter = transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3)
            img = color_jitter(img)
            
            if random.random() < 0.2:
                img = img.filter(ImageFilter.GaussianBlur(radius=1))
            
            # Ensure consistent size
            img = img.resize((self.width, self.height), Image.BILINEAR)
            label = label.resize((self.width, self.height), Image.NEAREST)
        
        else:
            img = img.resize((self.width, self.height), Image.BILINEAR)
            label = label.resize((self.width, self.height), Image.NEAREST)
        
        img = transforms.ToTensor()(img)
        img = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))(img)
        label = torch.tensor(np.array(label), dtype=torch.long)
        
        return img, label
    
    def __len__(self):
        return len(self.image_path)

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets, ignore_index=255):
        n_classes = inputs.shape[1]
        # mask valid pixels
        valid_mask = (targets != ignore_index)
        inputs = F.softmax(inputs, dim=1)
        # clamp targets for one-hot, then zero-out ignored locations
        safe_targets = torch.where(valid_mask, targets, torch.zeros_like(targets))
        targets_onehot = F.one_hot(safe_targets, num_classes=n_classes).permute(0, 3, 1, 2).float()
        valid_mask = valid_mask.unsqueeze(1).float()
        inputs = inputs * valid_mask
        targets_onehot = targets_onehot * valid_mask
        # flatten and compute dice over valid area only
        inputs = inputs.contiguous().view(-1, n_classes)
        targets_onehot = targets_onehot.contiguous().view(-1, n_classes)
        intersection = (inputs * targets_onehot).sum(dim=0)
        dice = (2. * intersection + self.smooth) / (inputs.sum(dim=0) + targets_onehot.sum(dim=0) + self.smooth)
        return 1 - dice.mean()

def train_model(data_dir='/kaggle/input/seg-data/seg_data'):
    # Hyperparameters
    batch_size = 8  # Adjusted for SegFormer memory
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    learning_rate = 1e-4  # Lower for pretrained
    num_epochs = 100
    start_epoch = 0
    best_mIoU = 0.0
    
    # Dataset with adjusted size
    train_data = ExpDataSet(data_dir, is_train=True)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, 
                            num_workers=4, pin_memory=True, drop_last=False)
    
    # Model: SegFormer B0 pretrained on Cityscapes
    segNet = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/segformer-b0-finetuned-cityscapes-768-768",
        num_labels=19,
        ignore_mismatched_sizes=True
    ).to(device)
    
    # Load existing best weights if available
    ckpt_path = './best_model.pth'
    if os.path.exists(ckpt_path):
        state = torch.load(ckpt_path, map_location=device)
        segNet.load_state_dict(state, strict=True)
     
    # Enable full encoder fine-tuning from the start
    for param in segNet.segformer.parameters():
        param.requires_grad = True
    
    # Mixed precision training
    scaler = torch.amp.GradScaler('cuda')
    
    # Compute class-balanced weights (ignore 255)
    class_weights = torch.ones(19, dtype=torch.float32).to(device)
    try:
        hist = np.zeros(19, dtype=np.int64)
        for p in train_data.label_path:
            lab = cv2.imread(p, 0)
            if lab is None:
                continue
            lab = lab[lab != 255]
            if lab.size > 0:
                hist += np.bincount(lab.flatten(), minlength=19)
        freq = hist / (hist.sum() + 1e-6)
        weights = 1.0 / (np.log(1.02 + freq) + 1e-8)
        class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
    except Exception:
        pass
    
    # Loss functions
    ce_loss = nn.CrossEntropyLoss(ignore_index=255, label_smoothing=0.02, weight=class_weights)
    dice_loss = DiceLoss()
    
    def combined_loss(outputs, targets):
        return 0.6 * ce_loss(outputs, targets) + 0.4 * dice_loss(outputs, targets)
    
    # Optimizer with discriminative learning rates (decoder higher LR, encoder lower LR)
    optimizer = torch.optim.AdamW([
        {"params": segNet.decode_head.parameters(), "lr": learning_rate, "weight_decay": 1e-2},
        {"params": segNet.segformer.parameters(),  "lr": learning_rate * 0.1, "weight_decay": 1e-2},
    ])
    
    # Scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs-start_epoch, eta_min=1e-6
    )
    
    # Exponential Moving Average (EMA) of weights
    ema_decay = 0.999
    ema_params = {name: p.detach().clone() for name, p in segNet.named_parameters() if p.requires_grad}
    
    # Training loop
    patience_counter = 0
    patience = 15
    mIoU_list = []
    
    for epoch in range(start_epoch, num_epochs):
        segNet.train()
        train_loss = 0.0
        optimizer.zero_grad()
        
        for iter, (imgs, labels) in enumerate(train_loader):
            imgs, labels = imgs.to(device), labels.long().to(device)
            
            with torch.amp.autocast('cuda'):
                outputs = segNet(imgs).logits
                outputs = F.interpolate(outputs, size=labels.shape[1:], mode='bilinear', align_corners=False)
                loss = combined_loss(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(segNet.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
            # EMA update
            with torch.no_grad():
                for name, p in segNet.named_parameters():
                    if p.requires_grad:
                        ema_params[name].mul_(ema_decay).add_(p.data, alpha=1.0 - ema_decay)
            
            train_loss += loss.item()
            
            if iter % 4 == 0:
                print(f'Epoch {epoch}/{num_epochs} Iter {iter}/{len(train_loader)} Loss={loss.item():.4f}')
            
            if iter % 10 == 0:
                torch.cuda.empty_cache()
        
        scheduler.step()
        
        avg_train_loss = train_loss / len(train_loader)
        print(f'Epoch {epoch} Average Training Loss: {avg_train_loss:.4f}')
        
        torch.cuda.empty_cache()
        gc.collect()
        
        # Evaluate every epoch with EMA weights to capture improvements and save the true best
        if (epoch + 1) % 5 == 0:
            segNet.eval()
            # Swap to EMA weights for evaluation
            backup_params = {name: p.detach().clone() for name, p in segNet.named_parameters()}
            with torch.no_grad():
                for name, p in segNet.named_parameters():
                    if name in ema_params:
                        p.copy_(ema_params[name])
            
            with torch.no_grad():
                gray_folder, color_folder = get_predictions(segNet, data_dir, device)
                current_mIoU = cal_acc(gray_folder, os.path.join(data_dir, 'testing/label'))
                mIoU_list.append(current_mIoU)
                print(f'Epoch {epoch} mIoU={current_mIoU:.4f}')
                
                if current_mIoU > best_mIoU:
                    best_mIoU = current_mIoU
                    patience_counter = 0
                    torch.save(segNet.state_dict(), './best_model.pth')
                    move_folders(
                        gray_folder, color_folder,
                        gray_folder.replace('temp_', ''),
                        color_folder.replace('temp_', '')
                    )
                    print(f'New best model saved with mIoU: {best_mIoU:.4f}')
                else:
                    patience_counter += 1
                    
                if os.path.exists(gray_folder):
                    shutil.rmtree(gray_folder)
                if os.path.exists(color_folder):
                    shutil.rmtree(color_folder)
            
            # Restore training weights after EMA eval
            with torch.no_grad():
                for name, p in segNet.named_parameters():
                    p.copy_(backup_params[name])
        
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch}. Best mIoU: {best_mIoU:.4f}')
            break
        
        torch.cuda.empty_cache()
        gc.collect()
    
    print(f'Training completed. Best mIoU: {best_mIoU:.4f}')
    return segNet, best_mIoU, mIoU_list

def calculate_gflops(model, input_size=None):
    input_size = (1, 3, 375, 1242)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device).eval()
    dummy = torch.randn(input_size, device=device)

    try:
        import logging
        from fvcore.nn import FlopCountAnalysis
        logging.getLogger("fvcore.nn.jit_analysis").setLevel(logging.ERROR)
        with torch.no_grad():
            flops = FlopCountAnalysis(model, dummy).total()
        return flops / 1e9
    except Exception as e:
        raise RuntimeError(
            "FLOPs computation failed: install fvcore."
        ) from e

if __name__ == "__main__":
    data_dir = '/kaggle/input/seg-data/seg_data'
    model, best_score, scores = train_model(data_dir)
    
    # Calculate GFLOPs
    model.eval()
    gflops = calculate_gflops(model)
    efficiency = best_score * 100 / gflops
    print(f"Final best mIoU: {best_score:.4f}")
    print(f"Efficiency Score: {efficiency:.2f}")

2025-08-21 03:51:48.787900: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755748308.810079     180 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755748308.816998     180 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


load info for 150 images
Epoch 0/100 Iter 0/19 Loss=1.1795
Epoch 0/100 Iter 4/19 Loss=1.2525
Epoch 0/100 Iter 8/19 Loss=1.1255
Epoch 0/100 Iter 12/19 Loss=1.1361
Epoch 0/100 Iter 16/19 Loss=1.1471
Epoch 0 Average Training Loss: 1.1978
Epoch 1/100 Iter 0/19 Loss=1.0192
Epoch 1/100 Iter 4/19 Loss=1.0284
Epoch 1/100 Iter 8/19 Loss=0.9071
Epoch 1/100 Iter 12/19 Loss=0.8877
Epoch 1/100 Iter 16/19 Loss=0.9258
Epoch 1 Average Training Loss: 1.0189
Epoch 2/100 Iter 0/19 Loss=1.0357
Epoch 2/100 Iter 4/19 Loss=0.9595
Epoch 2/100 Iter 8/19 Loss=0.9303
Epoch 2/100 Iter 12/19 Loss=0.9383
Epoch 2/100 Iter 16/19 Loss=0.9576
Epoch 2 Average Training Loss: 0.9534
Epoch 3/100 Iter 0/19 Loss=0.9159
Epoch 3/100 Iter 4/19 Loss=0.9429
Epoch 3/100 Iter 8/19 Loss=0.9347
Epoch 3/100 Iter 12/19 Loss=0.8744
Epoch 3/100 Iter 16/19 Loss=0.9944
Epoch 3 Average Training Loss: 0.8862
Epoch 4/100 Iter 0/19 Loss=0.9388
Epoch 4/100 Iter 4/19 Loss=0.8693
Epoch 4/100 Iter 8/19 Loss=0.8274
Epoch 4/100 Iter 12/19 Loss=0.873

In [5]:
# !pip install torch-pruning

In [6]:
# complete.py
# Aggressive structured pruning + KD fine-tuning to push SegFormer-B0 <4 GFLOPs on (1,3,375,1242).
# Requires: assignment_4.py in the same directory providing ExpDataSet, get_predictions, cal_acc, calculate_gflops.
# Kaggle paths: data_dir points to your dataset root; teacher checkpoint: ./best_model.pth

import os
import shutil
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

try:
    import torch_pruning as tp  # pip install torch-pruning==1.6.0
except ImportError as e:
    raise ImportError("Install torch-pruning 1.6.0: pip install torch-pruning==1.6.0") from e

from transformers import SegformerForSemanticSegmentation
# AMP helpers (torch.cuda.amp for CUDA; falls back to fp32 if not available)
try:
    from torch.cuda.amp import autocast as autocast_cuda
    from torch.cuda.amp import GradScaler as CudaGradScaler
    AMP_CUDA_AVAILABLE = torch.cuda.is_available()
except Exception:
    autocast_cuda = None
    CudaGradScaler = None
    AMP_CUDA_AVAILABLE = False


def build_model(ckpt_path: str, device: str) -> nn.Module:
    model = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/segformer-b0-finetuned-cityscapes-768-768",
        num_labels=19,
        ignore_mismatched_sizes=True,
    ).to(device)
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=True)
    return model


def find_encoder_mlp_dense1_by_stage(model: nn.Module):
    """
    Collect encoder MLP expansion linears 'mlp.dense1' grouped by encoder stage index (0..3).
    Names look like: 'encoder.block.0.0.mlp.dense1'
    """
    by_stage = {0: [], 1: [], 2: [], 3: []}
    for name, m in model.segformer.named_modules():
        if isinstance(m, nn.Linear) and ("mlp" in name and "dense1" in name):
            # Identify stage index from name if present
            stage_idx = None
            parts = name.split(".")
            for i, p in enumerate(parts):
                if p == "block" and i + 1 < len(parts):
                    try:
                        stage_idx = int(parts[i + 1])
                    except ValueError:
                        stage_idx = None
                    break
            if stage_idx in by_stage:
                by_stage[stage_idx].append(m)
    return by_stage


def prune_encoder_mlp_dense1_per_stage(model: nn.Module, device: str, stage_ratios=(0.20, 0.15, 0.10, 0.05), round_to: int = 8):
    """
    Prune only encoder MLP expansion 'dense1' layers with per-stage ratios.
    This reduces the intermediate width but keeps block input/output dims intact.
    """
    example_inputs = torch.randn(1, 3, 375, 1242).to(device)
    by_stage = find_encoder_mlp_dense1_by_stage(model)

    ratio_dict = {}
    for s, layers in by_stage.items():
        r = stage_ratios[s] if s in range(4) else 0.0
        for layer in layers:
            ratio_dict[layer] = r

    if not ratio_dict:
        return

    importance = tp.importance.GroupMagnitudeImportance(p=1)
    pruner = tp.pruner.BasePruner(
        model,
        example_inputs,
        importance=importance,
        pruning_ratio=0.0,             # dict-only pruning
        pruning_ratio_dict=ratio_dict,
        ignored_layers=[],              # allow dependency propagation into dwconv + dense2
        round_to=round_to,
    )
    pruner.step()


def prune_decoder_aggressive(model: nn.Module, device: str, linear_c_ratio: float = 0.70, linear_fuse_ratio: float = 0.80, round_to: int = 8):
    """
    Aggressively prune decoder:
      - linear_c[*].proj (Linear projections from encoder features): high ratio
      - linear_fuse (1x1 Conv2d on concatenated features): high ratio
    Exclude 'classifier' from direct pruning, but allow dependency updates to adjust its in_channels.
    """
    example_inputs = torch.randn(1, 3, 375, 1242).to(device)

    ratio_dict = {}
    for name, m in model.decode_head.named_modules():
        lname = name.lower()
        # Prune the 4 per-scale linear projections to decoder_hidden_size
        if isinstance(m, nn.Linear) and ("linear_c" in lname and "proj" in lname):
            ratio_dict[m] = linear_c_ratio
        # Heavily prune the fuse conv; this is a major GFLOPs contributor
        elif isinstance(m, nn.Conv2d) and ("linear_fuse" in lname):
            ratio_dict[m] = linear_fuse_ratio
        # Do not directly prune classifier; let dependencies update its in_channels only

    if not ratio_dict:
        return

    importance = tp.importance.GroupMagnitudeImportance(p=1)
    pruner = tp.pruner.BasePruner(
        model,
        example_inputs,
        importance=importance,
        pruning_ratio=0.0,
        pruning_ratio_dict=ratio_dict,
        ignored_layers=[],              # allow propagation across concat -> linear_fuse -> classifier
        round_to=round_to,
    )
    pruner.step()


def finetune_kd(model: nn.Module, data_dir: str, device: str,
                teacher_ckpt: str, epochs: int = 20,
                batch_size: int = 1, lr_dec: float = 3e-4, lr_enc: float = 1e-5,
                T: float = 5.0, alpha: float = 0.8):
    # Stabilize BN for batch_size=1
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

    # Teacher
    teacher = build_model(teacher_ckpt, device)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)

    train_ds = ExpDataSet(data_dir, is_train=True)
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=False,
    )

    ce_loss = nn.CrossEntropyLoss(ignore_index=255, label_smoothing=0.02)

    params = [
        {"params": model.decode_head.parameters(), "lr": lr_dec, "weight_decay": 1e-2},
        {"params": model.segformer.parameters(), "lr": lr_enc, "weight_decay": 1e-2},
    ]
    optimizer = torch.optim.AdamW(params)

    use_amp = AMP_CUDA_AVAILABLE and (autocast_cuda is not None) and (CudaGradScaler is not None)
    scaler = CudaGradScaler() if use_amp else None

    # Feature KD via decode_head.linear_fuse (pre-classifier fused features)
    beta = 0.2
    s_feat = [None]
    t_feat = [None]
    hook_handles = []

    def _make_hook(container):
        def hook(_, __, output):
            container[0] = output
        return hook

    s_fuse = None
    for name, m in model.decode_head.named_modules():
        if isinstance(m, nn.Conv2d) and ("linear_fuse" in name.lower()):
            s_fuse = m
            break
    t_fuse = None
    for name, m in teacher.decode_head.named_modules():
        if isinstance(m, nn.Conv2d) and ("linear_fuse" in name.lower()):
            t_fuse = m
            break
    if s_fuse is not None and t_fuse is not None:
        hook_handles.append(s_fuse.register_forward_hook(_make_hook(s_feat)))
        hook_handles.append(t_fuse.register_forward_hook(_make_hook(t_feat)))

    # If student/teacher fuse channels differ, learn a 1x1 projection for feature KD
    proj_s2t = None
    if s_fuse is not None and t_fuse is not None:
        s_ch = getattr(s_fuse, "out_channels", None)
        t_ch = getattr(t_fuse, "out_channels", None)
        if isinstance(s_ch, int) and isinstance(t_ch, int) and s_ch != t_ch:
            proj_s2t = nn.Conv2d(s_ch, t_ch, kernel_size=1, bias=True).to(device)

    # Add projection params to optimizer if present
    if proj_s2t is not None:
        optimizer.add_param_group({"params": proj_s2t.parameters(), "lr": lr_dec, "weight_decay": 0.0})

    # Exponential Moving Average of student weights
    ema_decay = 0.999
    ema_params = {name: p.detach().clone() for name, p in model.named_parameters() if p.requires_grad}

    model.train()
    for _ in range(epochs):
        for imgs, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.long().to(device, non_blocking=True)

            if use_amp:
                with autocast_cuda():
                    s_logits = model(imgs).logits
                    s_logits = F.interpolate(s_logits, size=labels.shape[1:], mode="bilinear", align_corners=False)

                    with torch.no_grad():
                        t_logits = teacher(imgs).logits
                        t_logits = F.interpolate(t_logits, size=labels.shape[1:], mode="bilinear", align_corners=False)

                    ce = ce_loss(s_logits, labels)
                    kd = F.kl_div(
                        F.log_softmax(s_logits / T, dim=1),
                        F.softmax(t_logits / T, dim=1),
                        reduction="batchmean",
                    ) * (T * T)
                    feat = 0.0
                    if s_feat[0] is not None and t_feat[0] is not None:
                        s_feat_cur = s_feat[0]
                        if proj_s2t is not None:
                            s_feat_cur = proj_s2t(s_feat_cur)
                        feat = F.mse_loss(s_feat_cur, t_feat[0].detach())
                    loss = alpha * kd + (1 - alpha) * ce + beta * feat

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            else:
                s_logits = model(imgs).logits
                s_logits = F.interpolate(s_logits, size=labels.shape[1:], mode="bilinear", align_corners=False)

                with torch.no_grad():
                    t_logits = teacher(imgs).logits
                    t_logits = F.interpolate(t_logits, size=labels.shape[1:], mode="bilinear", align_corners=False)

                ce = ce_loss(s_logits, labels)
                kd = F.kl_div(
                    F.log_softmax(s_logits / T, dim=1),
                    F.softmax(t_logits / T, dim=1),
                    reduction="batchmean",
                ) * (T * T)
                feat = 0.0
                if s_feat[0] is not None and t_feat[0] is not None:
                    s_feat_cur = s_feat[0]
                    if proj_s2t is not None:
                        s_feat_cur = proj_s2t(s_feat_cur)
                    feat = F.mse_loss(s_feat_cur, t_feat[0].detach())
                loss = alpha * kd + (1 - alpha) * ce + beta * feat

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            # EMA update after optimizer step
            with torch.no_grad():
                for name, p in model.named_parameters():
                    if p.requires_grad:
                        ema_params[name].mul_(ema_decay).add_(p.data, alpha=1.0 - ema_decay)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Remove hooks
    for h in hook_handles:
        try:
            h.remove()
        except Exception:
            pass

    # Load EMA weights into the student for evaluation
    with torch.no_grad():
        for name, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(ema_params[name])


def evaluate(model: nn.Module, data_dir: str, device: str):
    model.eval()
    with torch.no_grad():
        gray_folder, color_folder = get_predictions(model, data_dir, device)
        miou = cal_acc(gray_folder, os.path.join(data_dir, "testing/label"))
    shutil.rmtree(gray_folder, ignore_errors=True)
    shutil.rmtree(color_folder, ignore_errors=True)
    return miou


def main():
    torch.manual_seed(42)
    torch.backends.cudnn.benchmark = True

    device = "cuda" if torch.cuda.is_available() else "cpu"
    data_dir = "/kaggle/input/seg-data/seg_data"  # adjust if needed
    teacher_ckpt = "./best_model.pth"             # teacher for KD
    save_path = "./best_model_pruned_under4_kd.pth"

    # Build student initialized from teacher weights
    model = build_model(teacher_ckpt, device)

    # Baseline GFLOPs
    base_gflops = calculate_gflops(model)

    # Encoder MLP-expansion pruning per-stage (heavy early stages)
    prune_encoder_mlp_dense1_per_stage(
        model,
        device,
        stage_ratios=(0.20, 0.15, 0.10, 0.05),
        round_to=4,
    )

    # Decoder aggressive pruning: projections and fuse conv
    prune_decoder_aggressive(
        model,
        device,
        linear_c_ratio=0.72,   # 72% prune on linear_c[*].proj
        linear_fuse_ratio=0.82, # 82% prune on linear_fuse conv
        round_to=4,
    )

    # Pruned GFLOPs
    pruned_gflops = calculate_gflops(model)

    # KD fine-tuning
    finetune_kd(
        model,
        data_dir,
        device,
        teacher_ckpt=teacher_ckpt,
        epochs=30,
        batch_size=1,
        lr_dec=3e-4,
        lr_enc=1e-5,
        T=5.0,
        alpha=0.8,
    )

    # Evaluate
    miou = evaluate(model, data_dir, device)
    efficiency = miou * 100.0 / max(pruned_gflops, 1e-6)

    print(f"Baseline GFLOPs: {base_gflops:.2f}")
    print(f"Pruned GFLOPs: {pruned_gflops:.2f}")
    print(f"mIoU after pruning+KD: {miou:.4f}")
    print(f"Efficiency: {efficiency:.2f}")

    torch.save(model.state_dict(), save_path)


if __name__ == "__main__":
    main()

load info for 150 images
Begin testing
Eval result: mIoU 0.5475.
Baseline GFLOPs: 15.52
Pruned GFLOPs: 7.35
mIoU after pruning+KD: 0.5475
Efficiency: 7.45
